# PyDAP: OPeNDAP access as a request

OPeNDAP (DAP2/DAP4) serves slices of a remote dataset on demand.
`ea.plugins.pydap` is a request that returns either the OPeNDAP URLs to
open with xarray (`client="xarray"`, the default) or the native pydap
datasets (`client="pydap"`).

## Setup

```bash
pip install earthaccess jupyterlab matplotlib
pip install "earthaccess-plugin-pydap @ git+https://github.com/betolink/earthaccess-plugin-pydap"
```

All plugins except SlideRule use your Earthdata Login.


In [ ]:
import earthaccess as ea

ea.login()
ea.list_plugins(details=True)

## Search

One MUR Sea Surface Temperature (SST) granule.

In [ ]:
mursst = ea.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    temporal=("2023-06-01", "2023-06-02"),
    count=1,
)
len(mursst)

## What OPeNDAP offers for this collection

The links live in the granule UMM, so the check is local (no network):

In [ ]:
ea.plugins.pydap.capabilities(mursst)

## Inspect a granule's variables

`client="pydap"` returns the native pydap dataset — navigate the file and
see what variables are available (a metadata request, no data download).

In [ ]:
pyds = ea.plugins.pydap.request(mursst, client="pydap").results()[0]
pyds.tree()

## Request variables and open with xarray

`client="xarray"` yields **assets**: OPeNDAP DAP4 URLs with a
`?dap4.ce=...` constraint. `ea.open(job)` returns them for us to open.

In [ ]:
job = ea.plugins.pydap.request(mursst, variables=["analysed_sst", "lat", "lon", "time"])
urls = ea.open(job)
urls

In [ ]:
import xarray as xr

session = ea.get_requests_https_session()
ds = xr.open_dataset(urls[0], engine="pydap", session=session)
gulf_sst = ds["analysed_sst"].isel(time=0).sel(lon=slice(-98, -80), lat=slice(18, 31))
gulf_sst.plot(figsize=(10, 6), cmap="RdYlBu_r")

## Several granules

Keep the DAP4 constraint and open the assets together with the xarray kwargs
your data needs. Chunk to your subset so dask does not pull whole files:

```python
urls = ea.open(ea.plugins.pydap.request(many_granules, variables=["analysed_sst"]))
ds = xr.open_mfdataset(
    urls, engine="pydap", session=session, combine="nested",
    concat_dim="time", parallel=True, chunks={"lat": 400, "lon": 400},
)
```

## Download the constrained data

`ea.download(job, path)` serializes the constrained variables to local
NetCDF via pydap (one file per granule):

In [ ]:
# ea.download(job, "./data")

## Which plugins accept a dataset?

In [ ]:
mursst.list_services()

## Summary

- `request(results, client="pydap")` → native pydap datasets.
- `request(results, variables=[...])` → OPeNDAP URLs; `ea.open(job)` gives
  them to you, `ea.download(job, path)` writes local NetCDF.
- Open with xarray (`engine="pydap"`); earthaccess does not open for you.